# Задание 1 (синтаксис)

### Напишите функцию check_car_id(car_id: str) → str.
 
 Формат номера: 1 буква + 3 цифры + 2 буквы + 2–3 цифры региона.
 Буквы только из набора, допустимого в РФ-номерах: А, В, Е, К, М, Н, О, Р, С, Т, У, Х (кириллица). Учитывайте путаницу с латиницей на входе - нормализуйте или явно опишите правило.
 
 * Если номер валиден: «Номер А222ВС валиден. Регион: 96.»
 * Если невалиден: «Номер не валиден.»

Пример вывода:
```
>>> check_car_id('А222ВС96')
 Номер А222ВС валиден. Регион: 96.
 
 >>> check_car_id('А123ВЕ777')
 Номер А123ВЕ валиден. Регион: 777.
 
 >>> check_car_id('АБ22ВВ193')
 Номер не валиден.
```

Пограничные случаи:
```
>>> check_car_id('A222BC96')   # латиница A/B/C - по правилу нормализации
 Номер А222ВС валиден. Регион: 96.
 # либо «Номер не валиден.» - если нормализацию не делаете; зафиксируйте выбор
 
 >>> check_car_id('А222ВС9')    # регион из 1 цифры
 Номер не валиден.
 
 >>> check_car_id('А222ВС9699') # регион из 4 цифр
 Номер не валиден.
 
 >>> check_car_id('Я222ВС96')   # недопустимая буква Я
 Номер не валиден.
 
 >>> check_car_id('')
 Номер не валиден.
```

In [1]:
import re

def check_car_id(car_id: str) -> str:
    regexp = re.compile(r"[АВЕКМНОРСТУХ]{1}\d{3}[АВЕКМНОРСТУХ]{2}(\d{2,3})")
    car_id_match = re.match(regexp, car_id)

    if car_id_match:
        verdict = f"Номер {car_id} валиден. Регион: {car_id_match.group(1)}"
    else:
        verdict = "Номер не валиден."

    print(verdict)
    return verdict

print("--------- latin ")
check_car_id("M967AO51") #latin 
print("--------- latin/cyrillic")
check_car_id("М967AO51") #latin/cyrillic
print("--------- cyrillic")
check_car_id("М967АО51") #cyrillic

--------- latin 
Номер не валиден.
--------- latin/cyrillic
Номер не валиден.
--------- cyrillic
Номер М967АО51 валиден. Регион: 51


'Номер М967АО51 валиден. Регион: 51'

# Задание 2 (повышенная сложность)

### Дан многострочный лог. Реализуйте parse_logs(text: str) → list[dict].
 
 1. Одной (или составной) регуляркой извлеките строки только в формате:
    YYYY-MM-DD HH:MM:SS LEVEL ...
 2. Из хвоста строки достаньте user, action, ip; если есть card=... - тоже.
 3. Валидация через regex:
    • email (разумный практический паттерн);
    • IPv4;
    • карта: маска XXXX-XXXX-XXXX-XXXX; в результате хранить только ****-****-****-1111 (последние 4 цифры).
 4. Невалидные поля не роняют парсер: в словаре "errors": ["invalid_email", ...].
 5. Бонус: удалите последовательные повторы слов в текстовом поле сообщения через re.

Входной пример:

```
2024-03-15 14:22:01 INFO user=ivan@mail.ru action=login ip=192.168.0.12
 2024-03-15 14:22:05 WARN user=ivan@mail.ru action=retry ip=192.168.0.12
 2024-03-15 14:23:11 ERROR user=bad..email@ action=pay ip=999.1.1.1 card=4111-1111-1111-1111
 Mar 15 2024 14:24:00 INFO user=olga@company.com action=logout ip=10.0.0.5
```

 Пример вывода:
```
>>> parse_logs(text)
 [
   {
     'ts': '2024-03-15 14:22:01',
     'level': 'INFO',
     'user': 'ivan@mail.ru',
     'action': 'login',
     'ip': '192.168.0.12',
     'card': None,
     'errors': []
   },
   {
     'ts': '2024-03-15 14:22:05',
     'level': 'WARN',
     'user': 'ivan@mail.ru',
     'action': 'retry',
     'ip': '192.168.0.12',
     'card': None,
     'errors': []
   },
   {
     'ts': '2024-03-15 14:23:11',
     'level': 'ERROR',
     'user': 'bad..email@',
     'action': 'pay',
     'ip': '999.1.1.1',
     'card': '****-****-****-1111',
     'errors': ['invalid_email', 'invalid_ip']
   }
 ]
 ```
Строка «Mar 15 2024 ...» пропускается - формат даты не YYYY-MM-DD.

Пограничные случаи:

* Пустая строка / только пробелы → [].
* Строка без user=/action=/ip= → запись с отсутствующими полями (None) и соответствующими errors, либо пропуск - зафиксируйте правило.
* ip=256.0.0.1 или ip=1.2.3 → errors содержит invalid_ip.
* card=4111111111111111 без дефисов → не распознаётся как карта (card=None) или invalid_card — зафиксируйте правило.
* Повторы слов в сообщении: «error error error timeout» → «error timeout» (бонус).



In [ ]:
import re
import json

def parse_logs(text: str) -> list[dict]:
    text = re.sub(r"\b(\w+)(\s+\1\b)+", r"\1", text, flags=re.IGNORECASE) #убираем дубли
    regexp_main = re.compile(r"^(\d{4}-\d{1,2}-\d{1,2}\s?\d{1,2}:\d{1,2}:\d{1,2})\s?(CRITICAL|FATAL|ERROR|WARN|WARNING|INFO|DEBUG|NOTSET)\s?user=(.*?)\s?action=(.*?)\s?ip=([\.0-9]+)(?:\s?card=(.*))?$", flags=re.MULTILINE)
    logs_output = []
    for regexp_line in regexp_main.finditer(text):
        current_log = {"ts": regexp_line.group(1),
                       "level": regexp_line.group(2),
                       "user": regexp_line.group(3),
                       "action": regexp_line.group(4),
                       "ip": regexp_line.group(5)}
        errors = []
        if not re.match(r"^[\.a-zA-Z0-9]+\@[\.a-zA-Z0-9]+\.[a-zA-Z0-9]+$", regexp_line.group(3)):
            errors.append("invalid_email")
        if not re.match(r"^[a-zA-Z0-9\_]+$", regexp_line.group(4)):
            errors.append("invalid_action")
        if not re.match(r"^((25[0-5]|2[0-4][0-9]|1[0-9]{2}|[1-9][0-9]?|0)\.){3}(25[0-5]|2[0-4][0-9]|1[0-9]{2}|[1-9][0-9]?|0)$", regexp_line.group(5)):
            errors.append("invalid_ip")
        if regexp_line.group(6):
            card = re.match(r"^([0-9]{4})\-{1}([0-9]{4})\-{1}([0-9]{4})\-{1}([0-9]{4})$", regexp_line.group(6))
            if not card:
                errors.append("invalid_card")
            else:
                current_log["card"] = f"****-****-****-{card.group(4)}"
        else:
            current_log["card"] = "None"
        current_log["errors"] = errors
        logs_output.append(current_log)

    print(json.dumps(logs_output, ensure_ascii=False, indent=2))

    return logs_output

    
logs = """2024-03-15 14:22:01 INFO user=ivan@mail.ru action=login ip=192.168.0.12
2024-03-15 14:22:05 WARN user=ivan ivan@mail.ru action=retry ip=192.168.0.12
2024-03-15 14:23:11 ERROR user=bad..email@ action=pay pay ip=999.1.1.1 card=4111-1111-1111-1111
Mar 15 2024 14:24:00 INFO user=olga@company.com action=logout ip=10.0.0.5
2024-3-16 9:05:33 DEBUG user=dev@local action=ping ping ping ip=127.0.0.1
2024-03-16 09:06:12 CRITICAL user=admin@corp.ru action=shutdown ip=172.16.5.9 card=5500-0000-0000-0004
2024-03-16 09:07:45 FATAL user=admin@corp.ru action=panic ip=0.0.0.0
2024-3-16 9:08:01 NOTSET user=nobody nobody@ action=noop ip=255.255.255.255
2024-03-17 10:11:12 WARNING user=petr@site.org action=upload upload ip=8.8.8.8 card=3782-822463-10005
2024-03-17 10:12:59 INFO user=anna anna.smith@shop.net action=checkout ip=192.168.100.254
2024-03-17 10:14:00 ERROR user=broken@.com action=refund ip=300.1.2.3 card=4242-4242-4242-4242
2024-03-18 11:00:00 INFO user=x@x action=click click ip=1.1.1.1
2024-3-18 11:01:01 WARN user=test test@test action=warn ip=256.0.0.1
2024-03-18 11:02:02 DEBUG user=dev2@local action=trace ip=10.10.10.10 card=0000-0000-0000-0000
2024-03-18 11:03:03 INFO user=mail@sub.domain.co.uk action=login ip=203.0.113.7
2024-03-18 11:04:04 CRITICAL user=ops ops@infra action=alert alert alert ip=198.51.100.42 card=6011-0009-9013-9424"""

parsed_logs = parse_logs(logs)

[
  {
    "ts": "2024-03-15 14:22:01",
    "level": "INFO",
    "user": "ivan@mail.ru",
    "action": "login",
    "ip": "192.168.0.12",
    "card": "None",
    "errors": []
  },
  {
    "ts": "2024-03-15 14:22:05",
    "level": "WARN",
    "user": "ivan@mail.ru",
    "action": "retry",
    "ip": "192.168.0.12",
    "card": "None",
    "errors": []
  },
  {
    "ts": "2024-03-15 14:23:11",
    "level": "ERROR",
    "user": "bad..email@",
    "action": "pay",
    "ip": "999.1.1.1",
    "card": "****-****-****-1111",
    "errors": [
      "invalid_email",
      "invalid_ip"
    ]
  },
  {
    "ts": "2024-3-16 9:05:33",
    "level": "DEBUG",
    "user": "dev@local",
    "action": "ping",
    "ip": "127.0.0.1",
    "card": "None",
    "errors": [
      "invalid_email"
    ]
  },
  {
    "ts": "2024-03-16 09:06:12",
    "level": "CRITICAL",
    "user": "admin@corp.ru",
    "action": "shutdown",
    "ip": "172.16.5.9",
    "card": "****-****-****-0004",
    "errors": []
  },
  {
    "ts": "2